# Spatia Fine-tuning Pipeline theo cấu hình paper

Notebook này đóng gói pipeline huấn luyện Spatia end-to-end trên GPU, với thiết lập mini đúng yêu cầu đồ án: **100 video để train** và **10 video để test**, đều lấy từ RealEstate10K `test` split nhưng không trùng nhau.

Các bước chính:

1. Cài các thư viện cần thiết.
2. Kiểm tra CUDA, bật AMP/TF32/Flash Attention nếu GPU hỗ trợ.
3. Tải hoặc cache encoder model: Wan VAE và T5.
4. Tải metadata RealEstate10K và một pool video đủ lớn để tách train/test.
5. Preprocess video thành latent `.pt`.
6. Load dataset và model Spatia.
7. Huấn luyện Stage 1 và Stage 2 theo cấu hình paper.
8. Đánh giá trên 10 video test bằng các metric cùng tên với paper.

Mặc định `PRESET_NAME = "paper"`, tức dùng các hyperparameter bám sát paper. Các preset nhỏ hơn vẫn được giữ lại để debug nhanh khi máy không đủ VRAM.


## Đối chiếu với paper

Notebook đang bám paper ở các điểm quan trọng: dùng họ encoder Wan2.2/T5, huấn luyện bằng Flow Matching với timestep logit-normal, công thức nhiễu `x_t = (1 - t) * x_0 + t * x_T`, loss MSE trên velocity, 8 network blocks, mỗi block gồm 1 ControlNet và 4 main blocks, ControlNet được khởi tạo từ main block tương ứng, K=7 reference frames, target clip 81 frames, preceding clip 9 frames, Stage 1 train ControlNet 8.000 iterations với LR `1e-5`, Stage 2 fine-tune main blocks bằng LoRA rank 64 trong 5.000 iterations với LR `1e-4`, optimizer AdamW, batch size 64 và độ phân giải 720P.

Phần metric cũng được đặt theo paper trong phạm vi repo hiện có: RealEstate PSNR/SSIM/LPIPS, closed-loop PSNR_C/SSIM_C/LPIPS_C/Match Acc proxy, và ablation theo số reference frames K=1/3/5/7. Các metric WorldScore như Camera Control, Object Control, Style, Subject Quality hoặc Motion cần benchmark/evaluator bên ngoài nên notebook không tính trực tiếp.

Các điểm chưa thể giống paper hoàn toàn: repo hiện chưa có MapAnything, Keye-VL, ReferDINO, SAM2 và SLAM thật. Scene projection hiện vẫn là placeholder, reference retrieval dùng proxy cosine similarity thay vì 3D IoU trên point cloud, Match Acc dùng ORB proxy thay vì RoMa nếu chưa tích hợp RoMa, và backbone Spatia trong repo là implementation nội bộ chứ chưa load đầy đủ Wan2.2 5B transformer backbone.


## 0. Cài thư viện

Cell này cài các dependency trong `requirements.txt`. Nếu máy GPU đã có sẵn PyTorch CUDA đúng phiên bản, giữ `UPGRADE_TORCH = False` để tránh pip cài nhầm bản CPU hoặc thay đổi CUDA build đang hoạt động.


In [ ]:
INSTALL_DEPS = True
UPGRADE_TORCH = False

if INSTALL_DEPS:
    import importlib.util
    import subprocess
    import sys
    from pathlib import Path

    if importlib.util.find_spec("torch") is None:
        UPGRADE_TORCH = True

    req_path = Path("requirements.txt")
    lines = req_path.read_text(encoding="utf-8").splitlines()
    filtered = []
    for line in lines:
        stripped = line.strip().lower()
        if not UPGRADE_TORCH and stripped.startswith("torch"):
            continue
        filtered.append(line)

    tmp_req = Path(".notebook_requirements.txt")
    tmp_req.write_text("\n".join(filtered) + "\n", encoding="utf-8")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(tmp_req)])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ipywidgets"])
    print("Dependencies ready.")

## 1. Import, kiểm tra GPU và cấu hình môi trường

Notebook mặc định yêu cầu CUDA vì cấu hình paper rất nặng. Nếu chỉ muốn smoke test trên CPU, có thể đổi `REQUIRE_GPU = False`, nhưng train thật 100 video nên chạy trên GPU.

Cell này cũng bật TF32, cuDNN benchmark và Flash Attention nếu PyTorch/GPU hỗ trợ.


In [ ]:
import gc
import json
import math
import os
import random
import shutil
import sys
from pathlib import Path

import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader, Subset

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

from configs.config import SpatiaConfig
from data.dataset import SpatiaDataset
from models import Spatia
from pipeline.download import download_metadata, download_videos
from pipeline.preprocess import preprocess_all
from training.loss import flow_matching_loss
from utils.checkpoint import save_checkpoint

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

REQUIRE_GPU = True
if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required for this notebook. Move to a GPU runtime/machine first.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
    if hasattr(torch.backends.cuda, "enable_flash_sdp"):
        torch.backends.cuda.enable_flash_sdp(True)
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB VRAM)")
else:
    print("Running on CPU for smoke test only.")

if shutil.which("ffmpeg") is None:
    print("WARN: ffmpeg was not found on PATH. Install ffmpeg before downloading/preprocessing videos.")

In [ ]:
# Main knobs. Default is the paper profile.
TRAIN_VIDEOS = 100
TEST_VIDEOS = 10
REQUIRED_VIDEOS = TRAIN_VIDEOS + TEST_VIDEOS
# Download/process a slightly larger pool because some YouTube clips may be unavailable or too short.
# Only the first 100 processed samples are used for train and the next 10 for test.
DOWNLOAD_POOL_VIDEOS = 140
PRESET_NAME = "paper"  # options: paper, debug_gpu, gpu_16gb, gpu_24gb

PRESETS = {
    "paper": dict(
        height=720, width=1280, dim=1024, num_heads=16,
        num_network_blocks=8, num_sub_blocks=4, mlp_ratio=4.0,
        batch_size=64, lora_rank=64,
        t5_name="google/t5-v1_1-xxl", text_dim=4096,
        stage1_steps=8000, stage2_steps=5000,
        log_every=50,
    ),
    "debug_gpu": dict(
        height=96, width=128, dim=128, num_heads=4,
        num_network_blocks=1, num_sub_blocks=1, mlp_ratio=2.0,
        batch_size=1, lora_rank=8,
        t5_name="google/t5-v1_1-small", text_dim=512,
        stage1_steps=20, stage2_steps=20,
    ),
    "gpu_16gb": dict(
        height=128, width=128, dim=256, num_heads=4,
        num_network_blocks=2, num_sub_blocks=1, mlp_ratio=2.0,
        batch_size=1, lora_rank=16,
        t5_name="google/t5-v1_1-base", text_dim=768,
        stage1_steps=100, stage2_steps=100,
    ),
    "gpu_24gb": dict(
        height=160, width=224, dim=384, num_heads=6,
        num_network_blocks=3, num_sub_blocks=1, mlp_ratio=2.0,
        batch_size=1, lora_rank=16,
        t5_name="google/t5-v1_1-base", text_dim=768,
        stage1_steps=200, stage2_steps=200,
    ),
}

preset = PRESETS[PRESET_NAME]

RAW_DIR = Path("data/raw/realestate")
PROC_DIR = Path(
    f"data/processed_{PRESET_NAME}_{preset['height']}x{preset['width']}_t5{preset['text_dim']}"
)
SAVE_DIR = Path(f"checkpoints/{PRESET_NAME}_{TRAIN_VIDEOS}train_{TEST_VIDEOS}test")

VAE_NAME = "Wan-AI/Wan2.2-T2V-1.3B"
T5_NAME = preset["t5_name"]

cfg = SpatiaConfig()
cfg.height = preset["height"]
cfg.width = preset["width"]
cfg.dim = preset["dim"]
cfg.num_heads = preset["num_heads"]
cfg.num_main_blocks = preset["num_network_blocks"]
cfg.num_sub_blocks = preset["num_sub_blocks"]
cfg.mlp_ratio = preset["mlp_ratio"]
cfg.batch_size = preset["batch_size"]
cfg.lora_rank = preset["lora_rank"]
cfg.text_dim = preset["text_dim"]
cfg.stage1_iters = preset["stage1_steps"]
cfg.stage2_iters = preset["stage2_steps"]
cfg.log_every = preset.get("log_every", 50)
cfg.save_dir = str(SAVE_DIR)

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

h_lat = cfg.height // cfg.spatial_downsample
w_lat = cfg.width // cfg.spatial_downsample
n_t = (cfg.target_frames // cfg.temporal_downsample) * h_lat * w_lat
n_p = (cfg.preceding_frames // cfg.temporal_downsample) * h_lat * w_lat
n_r = cfg.max_ref_frames * h_lat * w_lat
if PRESET_NAME == "paper":
    print("WARN: paper profile follows the paper hyperparameters and expects very large multi-GPU memory (paper: 64 x AMD MI250).")
    print("WARN: this repo still uses its local Spatia implementation, not the full Wan2.2 5B transformer backbone.")

print(json.dumps({
    "preset": PRESET_NAME,
    "train_videos": TRAIN_VIDEOS,
    "test_videos": TEST_VIDEOS,
    "required_videos": REQUIRED_VIDEOS,
    "download_pool_videos": DOWNLOAD_POOL_VIDEOS,
    "device": str(DEVICE),
    "resolution": [cfg.height, cfg.width],
    "model_dim": cfg.dim,
    "heads": cfg.num_heads,
    "network_blocks": cfg.num_main_blocks,
    "sub_blocks": cfg.num_sub_blocks,
    "tokens_target": n_t,
    "tokens_total_attention": n_t + n_p + n_r,
    "proc_dir": str(PROC_DIR),
    "save_dir": str(SAVE_DIR),
}, indent=2))

## 2. Tải/cache và kiểm tra encoder model

Cell này tải hoặc cache các model cần cho preprocessing: Wan VAE và T5. Sau đó notebook load thử encoder để phát hiện sớm lỗi Hugging Face token, thiếu VRAM hoặc thiếu dependency.

Sau khi kiểm tra xong, model probe được giải phóng để nhường VRAM cho bước preprocess/train.


In [ ]:
DOWNLOAD_MODELS = True
CHECK_ENCODER_LOAD = True
HF_TOKEN = os.environ.get("HF_TOKEN")  # set this if the model repo requires auth

def hf_snapshot(repo_id: str, allow_patterns=None):
    from huggingface_hub import snapshot_download

    print(f"Caching {repo_id} ...")
    try:
        path = snapshot_download(
            repo_id=repo_id,
            allow_patterns=allow_patterns,
            token=HF_TOKEN,
        )
        print(f"  cached at: {path}")
        return path
    except Exception as exc:
        print(f"  WARN: could not pre-cache {repo_id}: {exc}")
        print("  The project encoder wrapper will try from_pretrained again during preprocessing.")
        return None

if DOWNLOAD_MODELS:
    hf_snapshot(VAE_NAME, allow_patterns=["vae/*", "model_index.json"])
    hf_snapshot(T5_NAME)

if CHECK_ENCODER_LOAD:
    from pipeline.encode import T5Encoder, WanVAE

    vae_probe = WanVAE(VAE_NAME, str(DEVICE))
    t5_probe = T5Encoder(T5_NAME, str(DEVICE))
    del vae_probe, t5_probe
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    print("Encoder load check done.")

## 3. Tải metadata RealEstate10K và video pool train/test

Dữ liệu được lấy từ RealEstate10K `test` split theo yêu cầu mini benchmark. Notebook cần tối thiểu 110 video hợp lệ: 100 video đầu dùng để train, 10 video tiếp theo dùng để test.

`DOWNLOAD_POOL_VIDEOS` được đặt lớn hơn 110 để dự phòng video YouTube bị lỗi, không tải được hoặc quá ngắn khi preprocess.


In [ ]:
RUN_DOWNLOAD = True
DOWNLOAD_WORKERS = 4
MAX_RETRIES = 3

def collect_video_pairs(meta_dir: Path, video_dir: Path, max_videos: int):
    pairs = []
    if not video_dir.exists():
        return pairs
    for vp in sorted(video_dir.glob("*.mp4")):
        tp = meta_dir / f"{vp.stem}.txt"
        pairs.append((vp, tp if tp.exists() else None))
    return pairs[:max_videos]

meta_dir = download_metadata(RAW_DIR)
video_dir = RAW_DIR / "videos"
video_dir.mkdir(parents=True, exist_ok=True)

existing_pairs = collect_video_pairs(meta_dir, video_dir, DOWNLOAD_POOL_VIDEOS)
print(f"Existing clips: {len(existing_pairs)}/{DOWNLOAD_POOL_VIDEOS} pool clips")

if RUN_DOWNLOAD and len(existing_pairs) < DOWNLOAD_POOL_VIDEOS:
    print("Downloading/resuming clips ...")
    download_videos(
        meta_dir=meta_dir,
        video_dir=video_dir,
        max_videos=DOWNLOAD_POOL_VIDEOS,
        height=cfg.height,
        workers=DOWNLOAD_WORKERS,
        max_retries=MAX_RETRIES,
    )

video_list = collect_video_pairs(meta_dir, video_dir, DOWNLOAD_POOL_VIDEOS)
print(f"Videos ready in pool: {len(video_list)}/{DOWNLOAD_POOL_VIDEOS}")
if len(video_list) == 0:
    raise RuntimeError("No videos found. Check yt-dlp/ffmpeg/network and rerun this cell.")
if len(video_list) < REQUIRED_VIDEOS:
    raise RuntimeError(f"Need at least {REQUIRED_VIDEOS} raw clips for {TRAIN_VIDEOS} train + {TEST_VIDEOS} test, got {len(video_list)}.")

## 4. Preprocess video thành latent `.pt`

Bước này dùng Wan VAE và T5 để chuyển từng video thành một file latent `.pt` trong `PROC_DIR`.

Mỗi sample lưu các tensor chính: `x_T`, `x_P`, `x_R`, `x_S_T`, `x_S_P` và `text`. Nếu notebook bị dừng giữa chừng, chạy lại cell này sẽ tự bỏ qua các file đã preprocess xong.


In [ ]:
RUN_PREPROCESS = True

if RUN_PREPROCESS:
    preprocess_all(
        video_list=video_list,
        proc_dir=PROC_DIR,
        cfg=cfg,
        device=str(DEVICE),
        vae_name=VAE_NAME,
        t5_name=T5_NAME,
    )
else:
    print("Skipping preprocessing.")

processed_files = sorted(PROC_DIR.glob("*.pt"))
print(f"Processed samples: {len(processed_files)} in {PROC_DIR}")
if len(processed_files) < REQUIRED_VIDEOS:
    raise RuntimeError(
        f"Need at least {REQUIRED_VIDEOS} processed samples "
        f"({TRAIN_VIDEOS} train + {TEST_VIDEOS} test), got {len(processed_files)}. "
        "Increase DOWNLOAD_POOL_VIDEOS and rerun download/preprocess."
    )

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

## 5. Tạo dataset và dataloader

Cell này load các file `.pt` đã preprocess và tách cố định thành:

- 100 sample đầu: tập train.
- 10 sample tiếp theo: tập test.

Hai tập này không trùng nhau. Notebook cũng in một vài tên file để dễ kiểm tra split trước khi train.


In [ ]:
NUM_WORKERS = 0  # safer in notebooks/Windows; raise to 2-4 on Linux if stable

dataset = SpatiaDataset(cfg, processed_dir=str(PROC_DIR))
if len(dataset) < REQUIRED_VIDEOS:
    raise RuntimeError(f"Need {REQUIRED_VIDEOS} processed samples, got {len(dataset)}.")

# Mini split requested by the assignment:
# - train: first 100 processed videos from RealEstate test split
# - test: next 10 processed videos from the same split, with no overlap
train_indices = list(range(0, TRAIN_VIDEOS))
test_indices = list(range(TRAIN_VIDEOS, REQUIRED_VIDEOS))
train_set = Subset(dataset, train_indices)
test_set = Subset(dataset, test_indices)
val_set = test_set

train_loader = DataLoader(
    train_set,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)
val_loader = DataLoader(
    test_set,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)

sample = next(iter(train_loader))
print(f"Train samples: {len(train_set)} | Test samples: {len(test_set)}")
if getattr(dataset, "is_real", False):
    print("Train files:", [p.name for p in dataset.real_files[:3]], "...")
    print("Test files:", [p.name for p in dataset.real_files[TRAIN_VIDEOS:TRAIN_VIDEOS + min(TEST_VIDEOS, 3)]], "...")
print({k: tuple(v.shape) for k, v in sample.items()})

## 6. Load model Spatia và chạy sanity check

Cell này khởi tạo model Spatia theo cấu hình đang chọn, đưa model lên GPU và in số lượng tham số.

Trước khi train, notebook chạy một forward pass nhỏ với noise để đảm bảo shape của batch, text token, scene token và reference token đều khớp với model.


In [ ]:
def move_batch(batch, device):
    return {k: v.to(device, non_blocking=True) for k, v in batch.items()}

def count_params(module):
    total = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    return total, trainable

model = Spatia(cfg).to(DEVICE)
total_params, trainable_params = count_params(model)
print(f"Model params: {total_params / 1e6:.2f}M | trainable: {trainable_params / 1e6:.2f}M")

AMP_DTYPE = torch.bfloat16 if (DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16
USE_AMP = DEVICE.type == "cuda"
print(f"AMP: {USE_AMP} | dtype: {AMP_DTYPE}")

model.eval()
batch = move_batch(sample, DEVICE)
with torch.no_grad(), torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
    t = torch.rand(batch["x_T"].shape[0], device=DEVICE)
    x_noise = torch.randn_like(batch["x_T"])
    out = model(
        x_noise,
        batch["x_P"],
        batch["x_R"],
        batch["x_S_T"],
        batch["x_S_P"],
        batch["text"],
        t,
    )

print(f"Forward OK: output={tuple(out.shape)} target={tuple(batch['x_T'].shape)}")
del batch, out, x_noise
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

## 7. Helper huấn luyện với AMP

Cell này định nghĩa các hàm phụ trợ cho training loop: tự lặp dataloader, bật mixed precision, gradient scaling, gradient clipping, scheduler step và log peak VRAM.

Các helper này được dùng chung cho Stage 1 và Stage 2.


In [ ]:
def make_scaler():
    enabled = USE_AMP and AMP_DTYPE == torch.float16
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except TypeError:
        return torch.cuda.amp.GradScaler(enabled=enabled)

def next_batch_forever(loader):
    while True:
        for batch in loader:
            yield batch

def train_steps(model, loader, optimizer, scheduler, max_steps, stage_name):
    model.train()
    scaler = make_scaler()
    stream = next_batch_forever(loader)
    total_loss = 0.0
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    for step in range(1, max_steps + 1):
        batch = move_batch(next(stream), DEVICE)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            loss = flow_matching_loss(model, batch, cfg, DEVICE)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], cfg.grad_clip
        )
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += float(loss.detach().cpu())
        if step == 1 or step % cfg.log_every == 0 or step == max_steps:
            lr = scheduler.get_last_lr()[0]
            msg = f"[{stage_name}] step {step:4d}/{max_steps} | loss={loss.item():.6f} | lr={lr:.2e}"
            if DEVICE.type == "cuda":
                mem = torch.cuda.max_memory_allocated() / 1024**3
                msg += f" | peak={mem:.2f}GB"
            print(msg)

    return total_loss / max_steps

## 8. Stage 1: Train ControlNet

Stage 1 bám paper: freeze main blocks và train phần ControlNet/projection/head với learning rate `1e-5`.

Sau khi chạy đủ số bước trong cấu hình, notebook lưu checkpoint `spatia_stage1.pt` vào `SAVE_DIR`.


In [ ]:
RUN_TRAIN = True

if RUN_TRAIN:
    print("Stage 1: freeze main blocks, train ControlNet/projections/head")
    model.freeze_main_blocks()
    total_params, trainable_params = count_params(model)
    print(f"Trainable params: {trainable_params / 1e6:.2f}M / {total_params / 1e6:.2f}M")

    opt1 = AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=cfg.lr_controlnet,
        weight_decay=cfg.weight_decay,
    )
    sch1 = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt1,
        T_max=max(cfg.stage1_iters, 1),
        eta_min=cfg.lr_controlnet * 0.1,
    )
    loss1 = train_steps(model, train_loader, opt1, sch1, cfg.stage1_iters, "Stage 1")
    stage1_path = save_checkpoint(model, opt1, 1, loss1, cfg.save_dir, "spatia_stage1")
    print(f"Stage 1 done. avg_loss={loss1:.6f} checkpoint={stage1_path}")
else:
    print("Skipping train.")

## 9. Stage 2: Fine-tune main blocks bằng LoRA

Stage 2 bám paper: bật LoRA rank 64 cho main blocks, freeze ControlNet và fine-tune main adapters với learning rate `1e-4`.

Checkpoint cuối được lưu dưới tên `spatia_final.pt`.


In [ ]:
if RUN_TRAIN:
    print("Stage 2: enable LoRA, freeze ControlNet, train main adapters")
    model.enable_lora()
    model = model.to(DEVICE)
    model.freeze_controlnet()
    model.unfreeze_main_blocks()
    total_params, trainable_params = count_params(model)
    print(f"Trainable params: {trainable_params / 1e6:.2f}M / {total_params / 1e6:.2f}M")

    opt2 = AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=cfg.lr_lora,
        weight_decay=cfg.weight_decay,
    )
    sch2 = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt2,
        T_max=max(cfg.stage2_iters, 1),
        eta_min=cfg.lr_lora * 0.1,
    )
    loss2 = train_steps(model, train_loader, opt2, sch2, cfg.stage2_iters, "Stage 2")
    final_path = save_checkpoint(model, opt2, 2, loss2, cfg.save_dir, "spatia_final")
    print(f"Stage 2 done. avg_loss={loss2:.6f} checkpoint={final_path}")
else:
    print("Skipping train.")

## 10. Đánh giá bằng metric kiểu paper

Paper report các nhóm metric chính: RealEstate PSNR/SSIM/LPIPS (Table 2), closed-loop memory PSNR_C/SSIM_C/LPIPS_C/Match Acc (Table 3), và ablation theo số reference frames K (Table 5).

Cell này tính các metric cùng tên trên 10 video test đã tách riêng. Một số thành phần được ghi rõ là proxy vì repo chưa có đầy đủ evaluator của paper: closed-loop camera trajectory thật của WorldScore, RoMa cho Match Accuracy và các metric WorldScore như Camera Control/Object Control/Style/Motion.

Kết quả được lưu thành JSON trong thư mục checkpoint để tiện đưa vào báo cáo.


In [ ]:
from torch.utils.data import DataLoader

from pipeline.encode import WanVAE
from training.evaluation import (
    build_lpips_model,
    evaluate_paper_metrics,
    write_metrics_json,
)

# Assignment mini benchmark: evaluate on the 10 held-out videos from RealEstate test split.
EVAL_MAX_SAMPLES = min(TEST_VIDEOS, len(test_set))
EVAL_BATCH_SIZE = 1
EVAL_ODE_STEPS = 20

# LPIPS at full 720P is closest to the paper but can be memory-heavy.
# Set LPIPS_RESIZE_TO = 256 only if your GPU runs out of memory during LPIPS.
LPIPS_RESIZE_TO = None

RUN_PIXEL_METRICS = True
REQUIRE_PIXEL_METRICS = True
RUN_REFERENCE_ABLATION = True  # Table-5 style K = 1/3/5/7
RUN_SCENE_REFERENCE_ABLATION = False  # Table-4 scaffold; enable if you need ablation.

eval_loader = DataLoader(
    test_set,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
)

metric_vae = None
lpips_model = None
if RUN_PIXEL_METRICS:
    metric_vae = WanVAE(VAE_NAME, str(DEVICE))
    if REQUIRE_PIXEL_METRICS and metric_vae.model is None:
        raise RuntimeError("Wan VAE must load successfully to compute PSNR/SSIM/LPIPS like the paper.")
    lpips_model = build_lpips_model(DEVICE)

main_metrics = evaluate_paper_metrics(
    model,
    eval_loader,
    cfg,
    device=DEVICE,
    vae=metric_vae,
    ode_steps=EVAL_ODE_STEPS,
    max_samples=EVAL_MAX_SAMPLES,
    use_amp=USE_AMP,
    amp_dtype=AMP_DTYPE,
    lpips_model=lpips_model,
    lpips_resize_to=LPIPS_RESIZE_TO,
)

metrics_report = {
    "paper_table_2_realestate": {
        "PSNR": main_metrics.get("realestate_psnr"),
        "SSIM": main_metrics.get("realestate_ssim"),
        "LPIPS": main_metrics.get("realestate_lpips"),
    },
    "paper_table_3_closed_loop_memory_proxy": {
        "PSNR_C": main_metrics.get("psnr_c"),
        "SSIM_C": main_metrics.get("ssim_c"),
        "LPIPS_C": main_metrics.get("lpips_c"),
        "Match_Acc": main_metrics.get("match_acc"),
    },
    "latent_sanity": {
        "Latent_PSNR": main_metrics.get("latent_psnr"),
    },
    "notes": [
        "Table 2 metrics compare decoded generated target video against decoded target video on the 10-video held-out RealEstate mini test set.",
        "Table 3 closed-loop metrics are a mini proxy: final generated frame vs decoded conditioning/preceding first frame. True paper setup requires WorldScore closed-loop camera trajectories.",
        "Match_Acc uses an ORB correspondence proxy unless a RoMa integration is added; the paper uses RoMa.",
    ],
}

if RUN_REFERENCE_ABLATION:
    table5 = []
    for k in [1, 3, 5, 7]:
        k_metrics = evaluate_paper_metrics(
            model,
            eval_loader,
            cfg,
            device=DEVICE,
            vae=metric_vae,
            ode_steps=EVAL_ODE_STEPS,
            max_samples=EVAL_MAX_SAMPLES,
            use_amp=USE_AMP,
            amp_dtype=AMP_DTYPE,
            ref_frames=k,
            lpips_model=lpips_model,
            lpips_resize_to=LPIPS_RESIZE_TO,
        )
        table5.append({
            "#Reference Frames": k,
            "PSNR_C": k_metrics.get("psnr_c"),
            "SSIM_C": k_metrics.get("ssim_c"),
            "LPIPS_C": k_metrics.get("lpips_c"),
            "Match_Acc": k_metrics.get("match_acc"),
        })
    metrics_report["paper_table_5_reference_frames"] = table5

if RUN_SCENE_REFERENCE_ABLATION:
    table4 = []
    for name, use_scene, use_ref in [
        ("no_scene_no_reference", False, False),
        ("scene_only", True, False),
        ("reference_only", False, True),
        ("scene_and_reference", True, True),
    ]:
        ab_metrics = evaluate_paper_metrics(
            model,
            eval_loader,
            cfg,
            device=DEVICE,
            vae=metric_vae,
            ode_steps=EVAL_ODE_STEPS,
            max_samples=EVAL_MAX_SAMPLES,
            use_amp=USE_AMP,
            amp_dtype=AMP_DTYPE,
            use_scene=use_scene,
            use_reference=use_ref,
            lpips_model=lpips_model,
            lpips_resize_to=LPIPS_RESIZE_TO,
        )
        table4.append({
            "Variant": name,
            "PSNR_C": ab_metrics.get("psnr_c"),
            "SSIM_C": ab_metrics.get("ssim_c"),
            "LPIPS_C": ab_metrics.get("lpips_c"),
            "Match_Acc": ab_metrics.get("match_acc"),
        })
    metrics_report["paper_table_4_scene_reference_ablation"] = table4

metrics_path = SAVE_DIR / "paper_style_metrics_100train_10test.json"
write_metrics_json(metrics_path, metrics_report)
print(json.dumps(metrics_report, indent=2))
print(f"Metrics saved to: {metrics_path}")
print(f"Done. checkpoints saved in: {cfg.save_dir}")